# 1. COMET Image Preprocessing and Nimbus Prediction Workflow

This notebook runs the COMET slide-level workflow from raw multiplex whole-slide images to deduplicated cell masks and NIMBUS marker probabilities.

## Stage map

1. **Tile whole-slide images** into overlapping FOVs while excluding low-information background tiles.
2. **Extract marker channels** from each multi-channel FOV into named single-channel TIFFs.
3. **Fuse nuclear and membrane signals** into 2-channel CellposeSAM inputs.
4. **Segment cells with CellposeSAM** on the fused inputs.
5. **Deduplicate overlapping masks** so each biological cell is counted once across neighboring tiles.
6. **Score marker positivity with NIMBUS** using the per-marker TIFFs and the final masks.

## Expected experiment layout

```text
../data/Example
|-- Slide1.ome.tif
|-- Slide2.ome.tif
|-- ...
```

After Stage 1, COMET creates one Sample folder per raw image and writes the following structure:

```text
..data/Example/
|-- Slide1.ome.tif
|-- Slide1/
    |-- fov_coordinates.csv
    |-- Tiles/
    |   |-- FOV0.tif, FOV1.tif, ...
    |-- image_data/
    |   |-- FOV0/
    |       |-- DAPI.tif
    |       |-- CD3.tif
    |       |-- ...
    |-- segmentation/
    |   |-- cellpose_input/
    |   |-- cellpose_output/
    |   |-- deepcell_output/
    |   |-- deepcell_output_bak/
    |-- nimbus_output/
```

## Tips

- Review `CHANNEL_NAMES` carefully before channel extraction; the order must match the acquisition order in the raw OME-TIFF metadata.
- If you have connection error to download Nimbus prediction model V1.pt, please download it from the release and copy into /YOUR/CONDA/envs/comet/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt


## Stage 1. Image Preparation

Stage 1 is intentionally split into two notebook steps. First, inspect the OME metadata from the first raw slide so you can confirm the acquisition order. Second, fill in `CHANNEL_NAMES` manually and run the preprocessing pipeline to tile slides, extract marker TIFFs, and build 2-channel CellposeSAM inputs.

`CHANNEL_NAMES` must follow the raw acquisition order reported by the metadata inspection cell. You may provide only the first `N` names if you want COMET to export the first `N` raw channels and ignore the remaining trailing channels.

### Input

```text
<BASE_DIR>/*.ome.tif
```

### Outputs

```text
<BASE_DIR>/<slide>/Tiles/FOV*.tif
<BASE_DIR>/<slide>/image_data/FOV*/<marker>.tif
<BASE_DIR>/<slide>/segmentation/cellpose_input/FOV*.tiff
```


In [ ]:
# Cell strategy:
# Bootstrap the local COMET source tree into sys.path so the notebook can
# import from the repository even when Jupyter starts from a different cwd.

import sys
from pathlib import Path

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "COMET",
]

for candidate in CANDIDATE_ROOTS:
    if (candidate / "comet").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not locate the COMET package directory. "
        "Open the notebook from the repository or add the package root to sys.path."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Using COMET source from: {PROJECT_ROOT}")

from comet import inspect_experiment_metadata, run_signal_preparation_pipeline


In [ ]:
# Cell strategy:
# Step 1 of Stage 1: point to the experiment directory and inspect the
# metadata of the first raw slide. Use the printed order to decide how to
# fill CHANNEL_NAMES in the next cell.

BASE_DIR = PROJECT_ROOT / "data" / "example"
SLIDE_SUFFIX = ".ome.tif"

if BASE_DIR.exists():
    try:
        inspect_experiment_metadata(
            base_dir=BASE_DIR,
            slide_suffix=SLIDE_SUFFIX,
        )
    except FileNotFoundError as e:
        print(f"  [SKIP] Metadata inspection could not run: {e}")
else:
    print(f"BASE_DIR {BASE_DIR} does not exist. Please check the path.")


In [ ]:
# Cell strategy:
# Step 2 of Stage 1: after reviewing the previous cell output, edit
# CHANNEL_NAMES so they match the metadata order you want to keep.
# You may provide only the first N names to ignore trailing channels.

CHANNEL_NAMES = [
    "DAPI",
    "CD8a",
    "EOMES",
    "TCRbeta",
    "CD45",
    "CD4",
    "CD3",
]
NUCLEAR_MARKERS = ["DAPI", "EOMES"]
MEMBRANE_MARKERS = ["CD8a", "TCRbeta", "CD45", "CD4", "CD3"]
NIMBUS_CHANNELS = ["CD8a", "EOMES", "TCRbeta", "CD45", "CD4", "CD3"]

TILE_SIZE = 1024
OVERLAP = 102
TISSUE_THRESHOLD = 10.0 # 10% of FOV size
NORMALIZE_FOR_CELLPOSE = True


In [ ]:
# Cell strategy:
# Step 3 of Stage 1: run tiling, channel extraction, and Cellpose input
# preparation after CHANNEL_NAMES has been confirmed by the user.

if BASE_DIR.exists():
    try:
        run_signal_preparation_pipeline(
            base_dir=BASE_DIR,
            channel_names=CHANNEL_NAMES,
            nuclear_markers=NUCLEAR_MARKERS,
            membrane_markers=MEMBRANE_MARKERS,
            slide_suffix=SLIDE_SUFFIX,
            tile_size=TILE_SIZE,
            overlap=OVERLAP,
            tissue_threshold=TISSUE_THRESHOLD,
            normalize=NORMALIZE_FOR_CELLPOSE,
        )
    except (FileNotFoundError, ValueError) as e:
        print(f"  [SKIP] Stage 1 could not run: {e}")
else:
    print(f"BASE_DIR {BASE_DIR} does not exist. Please check the path.")


## Stage 2. CellposeSAM Segmentation

This stage consumes the fused 2-channel TIFFs created in Stage 1 and runs CellposeSAM to generate whole-cell masks for each FOV. In biological terms, the nuclear channel anchors cell centers and the membrane channel improves cell boundary delineation.

### Input

```text
<slide>/segmentation/cellpose_input/FOV*.tiff
```

### Output

```text
<slide>/segmentation/cellpose_output/      # raw Cellpose artifacts
<slide>/segmentation/deepcell_output/      # renamed whole-cell masks for Nimbus input
```

The next stage assumes masks are named `FOV*_whole_cell.tiff` under `segmentation/deepcell_output/`.


In [ ]:
# Cell strategy:
# Import the CellposeSAM batch wrapper and the standard libraries typically
# needed for segmentation bookkeeping and debugging.

import os
import re
import glob
import time
import shutil
from tqdm import tqdm
from comet import run_cellpose_experiment


In [ ]:
# Cell strategy:
# Run CellposeSAM for all slide folders under BASE_DIR.
# Reads:  <slide>/segmentation/cellpose_input/FOV*.tiff
# Writes: <slide>/segmentation/deepcell_output/FOV*_whole_cell.tiff

BATCH_SIZE = 32

if Path(BASE_DIR).exists():
    print(f"==== Start processing Cellpose segmentation in: {BASE_DIR} ====")

    run_cellpose_experiment(
        experiment_dir=BASE_DIR,
        slide_list=None,
        channels=[1, 0],
        diameter=None,
        flow_threshold=0.8,
        cellprob_threshold=0.0,
        batch_size=BATCH_SIZE,
        model_type="cpsam",
        input_subdir="cellpose_input",
        raw_subdir="cellpose_output",
        final_subdir="deepcell_output",
    )
    print("==== All slide folders finished CellposeSAM segmentation ====")
else:
    print(f"Sample root directory {BASE_DIR} does not exist. Please check the path.")


## Stage 3. Mask Deduplication Across Overlapping Tiles

Whole-slide tissue is processed in overlapping FOVs, so the same biological cell can be segmented twice near tile borders. This stage first removes incomplete edge-touching masks and then resolves duplicated instances on a virtual stitched canvas.

### Input

```text
<slide>/fov_coordinates.csv
<slide>/segmentation/deepcell_output/FOV*_whole_cell.tiff
```

### Output

```text
<slide>/segmentation/deepcell_output/      # deduplicated masks in place
<slide>/segmentation/deepcell_output_bak/  # backup before overlap resolution
```

This is the mask set that should be forwarded to NIMBUS for marker quantification.


In [ ]:
# Cell strategy:
# Import the overlap-resolution utilities used to clean Cellpose outputs
# before downstream marker scoring.

import warnings
import pandas as pd
from skimage.segmentation import clear_border
from skimage.measure import regionprops
from tqdm import tqdm
from typing import Optional
from comet import deduplicate_experiment


In [ ]:
# Cell strategy:
# Deduplicate masks across all slide folders. The default settings keep
# the current COMET behavior: border clearing + overlap-aware stitching.

if BASE_DIR.exists():
    print(f"==== Start deduplication pipeline in {BASE_DIR} ====")

    deduplicate_experiment(
        experiment_dir=str(BASE_DIR),
        slide_list=None,
        overlap_threshold=0.1,
        min_cell_size=0,
        tiff_pattern="{FOV_Name}_whole_cell.tiff",
        clear_borders=True,
        border_buffer=2,
        seg_subdir="deepcell_output",
    )

    print("==== All slide folders finished deduplication ====")
else:
    print(f"Sample root directory {BASE_DIR} does not exist. Please check the path.")


## Stage 4. NIMBUS Marker Probability Inference

NIMBUS combines the final segmentation masks with the per-marker TIFF stack to estimate marker positivity for each segmented cell. In practice, this is the bridge between morphology-aware segmentation and downstream cell state annotation.

### Input

```text
<slide>/image_data/FOV*/<marker>.tif                          # per-marker TIFF
<slide>/segmentation/deepcell_output/FOV*_whole_cell.tiff     # cell segmentation masks
```

### Output

```text
<slide>/nimbus_output/nimbus_cell_table.csv
```

The `include_channels` list below should match the marker names present under each `image_data/FOV*/` directory.


In [ ]:
# Cell strategy:
# Import the NIMBUS experiment wrapper used for marker-level probability
# scoring on the final, deduplicated masks.

from comet import run_nimbus_experiment


In [ ]:
# Cell strategy:
# Run NIMBUS on the deduplicated masks and the named marker TIFF files.
# Reads:  <slide>/image_data/ and <slide>/segmentation/deepcell_output/
# Writes: <slide>/nimbus_output/nimbus_cell_table.csv

if BASE_DIR.exists():
    print(f"==== Start marker prediction in {BASE_DIR} ====")

    run_nimbus_experiment(
        experiment_dir=str(BASE_DIR),
        include_channels=NIMBUS_CHANNELS,
        slide_list=None,
    )

    print("==== All slide folders finished NIMBUS inference ====")
else:
    print(f"Sample root directory {BASE_DIR} does not exist. Please check the path.")


## QC Notes and Hand-off Checklist

- Confirm the channel order reported by `print_channel_metadata` matches `CHANNEL_NAMES` before trusting Stage 1 outputs.
- Spot check `Tiles/` to verify that tissue-rich regions were retained and blank background tiles were excluded.
- Spot check `image_data/FOV*/` to confirm marker filenames are biologically meaningful and consistent with antibody naming.
- Review a few `segmentation/deepcell_output/FOV*_whole_cell.tiff` masks before running large-scale NIMBUS inference.
- Once `nimbus_cell_table.csv` is generated, the dataset is ready for thresholding, cell type assignment, and QuPath export.
